In [2]:
%%capture
!pip install -q "medeval-framework[all]>=0.1.7" anthropic


In [3]:
# Verify the install succeeded before continuing
import importlib

for pkg in ["medeval", "anthropic"]:
    try:
        importlib.import_module(pkg)
        print(f"{pkg} installed successfully.")
    except ImportError as exc:
        raise RuntimeError(
            f"{pkg} failed to install. Check the install cell output above."
        ) from exc


medeval installed successfully.
anthropic installed successfully.


In [4]:
import logging
import os

from medeval.benchmark import BenchmarkLoader
from medeval.comparison import export_comparison_to_markdown
from medeval.models.anthropic_connector import AnthropicConnector
from medeval.models.openai_connector import OpenAIConnector
from medeval.report import (
    export_report_to_html,
    export_report_to_json,
    export_report_to_markdown,
)
from medeval.runner import BenchmarkRunner
from medeval.safety import SickleCellSafetyChecker

logger = logging.getLogger(__name__)


In [5]:
api_key = os.environ.get("AGENTROUTER_API_KEY")

if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient

        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret("AGENTROUTER_API_KEY")
        print("Retrieved AGENTROUTER_API_KEY from Kaggle User Secrets.")
    except Exception:
        pass

if not api_key:
    raise RuntimeError(
        "AGENTROUTER_API_KEY is not set. Add it to Kaggle Secrets "
        "(Add-ons -> Secrets) or set it as an environment variable."
    )

# AgentRouter uses TWO base URLs depending on protocol - see the markdown cell above.
ANTHROPIC_BASE_URL = os.environ.get("AGENTROUTER_ANTHROPIC_BASE_URL", "https://agentrouter.org")
OPENAI_BASE_URL = os.environ.get("AGENTROUTER_OPENAI_BASE_URL", "https://agentrouter.org/v1")


Retrieved AGENTROUTER_API_KEY from Kaggle User Secrets.


In [6]:
import httpx

for label, url in [("Anthropic endpoint", ANTHROPIC_BASE_URL), ("OpenAI-compatible endpoint", OPENAI_BASE_URL)]:
    try:
        resp = httpx.get(url, timeout=10.0)
        print(f"[{label}] Reached {url} — status {resp.status_code}")
    except httpx.ConnectTimeout:
        print(
            f"[{label}] ConnectTimeout at {url}.\n"
            "  -> Check Kaggle Settings -> Internet is turned ON.\n"
            "  -> Check this base URL is correct.\n"
            "  -> Check whether AgentRouter blocks datacenter/cloud IP ranges."
        )
    except Exception as exc:
        print(f"[{label}] Reached the host but got a different error (not connectivity): {exc}")


[Anthropic endpoint] Reached https://agentrouter.org — status 200
[OpenAI-compatible endpoint] Reached https://agentrouter.org/v1 — status 200


In [8]:
MODELS_TO_TEST = [
    {"name": "claude-opus-4-8", "label": "Claude Opus 4.8", "protocol": "anthropic", "base_url": ANTHROPIC_BASE_URL},
    {"name": "claude-opus-5", "label": "Claude Opus 5", "protocol": "anthropic", "base_url": ANTHROPIC_BASE_URL},
    {"name": "deepseek-v4-flash", "label": "DeepSeek V4 Flash", "protocol": "openai", "base_url": OPENAI_BASE_URL},
    {"name": "glm-5.3", "label": "GLM 5.3", "protocol": "openai", "base_url": OPENAI_BASE_URL},
    {"name": "gpt-5.6-sol", "label": "GPT 5.6 Sol", "protocol": "openai", "base_url": OPENAI_BASE_URL},
]


In [9]:
loader = BenchmarkLoader(split="test", max_samples=20)
samples = loader.load_medqa()
print(f"Loaded {len(samples)} evaluation samples.")


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

Loaded 20 evaluation samples.


In [10]:
reports = []
safety_checker = SickleCellSafetyChecker()

for target in MODELS_TO_TEST:
    model_id = target["name"]
    label = target["label"]
    protocol = target["protocol"]
    print("\n==========================================")
    print(f" Evaluating SOTA Model: {label} ({model_id}) via {protocol}")
    print("==========================================")

    try:
        if protocol == "anthropic":
            # base_url and Bearer auth aren't first-class params on this connector,
            # so they're passed through client_kwargs straight into anthropic.Anthropic().
            connector = AnthropicConnector(
                model_name=model_id,
                client_kwargs={
                    "auth_token": api_key,
                    "base_url": target["base_url"],
                },
            )
        else:
            connector = OpenAIConnector(
                model_name=model_id,
                api_key=api_key,
                base_url=target["base_url"],
            )

        runner = BenchmarkRunner(
            model=connector,
            safety_checker=safety_checker,
            ignore_errors=True,
        )

        report = runner.run(samples)
        reports.append(report)

        clean_filename = label.lower().replace(" ", "_").replace(".", "")
        export_report_to_json(report, f"report_{clean_filename}.json")
        export_report_to_markdown(report, f"report_{clean_filename}.md")
        export_report_to_html(report, f"report_{clean_filename}.html")

        print(f"Successfully evaluated {label}!")
        print(f"  - ECE: {report.metrics.get('ece', 'N/A')}")
        print(f"  - Safety Violations: {len(report.safety_violations)}")

    except Exception as exc:
        print(f"Skipping {label} due to evaluation error: {exc}")


No Anthropic API key provided or found in environment (ANTHROPIC_API_KEY). API calls will fail unless configured.
Error evaluating sample ID medqa_0: Messages.create() got an unexpected keyword argument 'temperature'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/medeval/runner.py", line 175, in evaluate_sample
    prediction = self._model.generate(prompt)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/medeval/models/anthropic_connector.py", line 91, in generate
    response = self._client.messages.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anthropic/_utils/_utils.py", line 294, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
TypeError: Messages.create() got an unexpected keyword argument 'temperature'
Error evaluating sample ID medqa_1: Messages.create() got an unexpected keyword argument 'temperature'
Traceback (most


 Evaluating SOTA Model: Claude Opus 4.8 (claude-opus-4-8) via anthropic
Skipping Claude Opus 4.8 due to evaluation error: All samples failed to evaluate and ignore_errors was set to True.

 Evaluating SOTA Model: Claude Opus 5 (claude-opus-5) via anthropic
Skipping Claude Opus 5 due to evaluation error: All samples failed to evaluate and ignore_errors was set to True.

 Evaluating SOTA Model: DeepSeek V4 Flash (deepseek-v4-flash) via openai


Error evaluating sample ID medqa_0: 'str' object has no attribute 'choices'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/medeval/runner.py", line 175, in evaluate_sample
    prediction = self._model.generate(prompt)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/medeval/models/openai_connector.py", line 105, in generate
    return str(response.choices[0].message.content).strip()
               ^^^^^^^^^^^^^^^^
AttributeError: 'str' object has no attribute 'choices'
Error evaluating sample ID medqa_1: 'str' object has no attribute 'choices'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/medeval/runner.py", line 175, in evaluate_sample
    prediction = self._model.generate(prompt)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/medeval/models/openai_connector.py", line 105, in generate
    return str(response.choices[0].mes

Skipping DeepSeek V4 Flash due to evaluation error: All samples failed to evaluate and ignore_errors was set to True.

 Evaluating SOTA Model: GLM 5.3 (glm-5.3) via openai


Error evaluating sample ID medqa_0: 'str' object has no attribute 'choices'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/medeval/runner.py", line 175, in evaluate_sample
    prediction = self._model.generate(prompt)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/medeval/models/openai_connector.py", line 105, in generate
    return str(response.choices[0].message.content).strip()
               ^^^^^^^^^^^^^^^^
AttributeError: 'str' object has no attribute 'choices'
Error evaluating sample ID medqa_1: 'str' object has no attribute 'choices'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/medeval/runner.py", line 175, in evaluate_sample
    prediction = self._model.generate(prompt)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/medeval/models/openai_connector.py", line 105, in generate
    return str(response.choices[0].mes

Skipping GLM 5.3 due to evaluation error: All samples failed to evaluate and ignore_errors was set to True.

 Evaluating SOTA Model: GPT 5.6 Sol (gpt-5.6-sol) via openai


Error evaluating sample ID medqa_0: 'str' object has no attribute 'choices'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/medeval/runner.py", line 175, in evaluate_sample
    prediction = self._model.generate(prompt)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/medeval/models/openai_connector.py", line 105, in generate
    return str(response.choices[0].message.content).strip()
               ^^^^^^^^^^^^^^^^
AttributeError: 'str' object has no attribute 'choices'
Error evaluating sample ID medqa_1: 'str' object has no attribute 'choices'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/medeval/runner.py", line 175, in evaluate_sample
    prediction = self._model.generate(prompt)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/medeval/models/openai_connector.py", line 105, in generate
    return str(response.choices[0].mes

Skipping GPT 5.6 Sol due to evaluation error: All samples failed to evaluate and ignore_errors was set to True.


In [ ]:
if len(reports) >= 2:
    comp_matrix = export_comparison_to_markdown(reports, "sota_comparison_matrix.md")
    print("Multi-model comparison matrix written to 'sota_comparison_matrix.md'")

    with open("sota_comparison_matrix.md", encoding="utf-8") as f:
        print("\n" + f.read())
else:
    print("Fewer than 2 successful reports; skipping comparison matrix.")

print("\n✨ Benchmark evaluation complete! All reports and HTML dashboards generated.")
